# 2.3 Le plancher de contrôle
**Sujet 04 Allstate · P3 · HOUNGBO Gwenaël**

Question : si l'on n'a aucun modèle, **quelle valeur unique annoncer**, et quel score cela donne-t-il **en validation, donc honnêtement** ?
Ce score est le *plancher* : tout modèle doit le battre pour être utile.

Différence avec 1-10 : en 1-10 on mesure la constante sur tout le train. Ici on la mesure **comme on mesurera un modèle** : la constante est calculée sur le train du pli, et le score sur la validation du pli.

*À lancer depuis le dossier du projet (celui qui contient `data/`).*

## Chargement

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("data")          # dossier qui contient train.csv
OUT = Path("sorties")        # dossier des résultats
OUT.mkdir(exist_ok=True)

train = pd.read_csv(DATA / "train.csv")
print(train.shape)           # attendu : (150654, 132)

cat_cols = [c for c in train.columns if c.startswith("cat")]
cont_cols = [c for c in train.columns if c.startswith("cont")]
y = train["loss"]

## Le découpage de validation

In [ ]:
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5

# Découpage : on range les montants en 10 tranches (déciles) et chaque pli reçoit
# la même proportion de chaque tranche. Ainsi la queue des gros sinistres,
# rare, se retrouve dans tous les plis.
# >>> Si l'équipe a fixé un autre découpage en 2.1, remplace ces lignes par le sien. <<<
tranches = pd.qcut(y, q=10, labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
plis = list(skf.split(np.zeros(len(y)), tranches))      # liste de (indices train, indices validation)
print(len(plis), "plis |", len(plis[0][0]), "lignes d'entraînement et", len(plis[0][1]), "de validation par pli")

## Plancher mesuré pli par pli

`DummyRegressor` est un « modèle » qui annonce toujours la même valeur :
- `strategy="mean"` : la moyenne du train du pli
- `strategy="median"` : la médiane du train du pli

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

resultats = []
constantes = {"mean": [], "median": []}
for strategie in ["mean", "median"]:
    maes = []
    for itr, iva in plis:
        modele = DummyRegressor(strategy=strategie)
        modele.fit(np.zeros((len(itr), 1)), y.iloc[itr])        # apprend la constante sur le train du pli
        pred = modele.predict(np.zeros((len(iva), 1)))           # annonce cette constante partout
        maes.append(mean_absolute_error(y.iloc[iva], pred))      # MAE sur la validation du pli
        constantes[strategie].append(float(modele.constant_[0][0]))
    resultats.append({"constante": strategie, "mae_cv": round(np.mean(maes), 2), "ecart_type": round(np.std(maes), 2)})
plancher = pd.DataFrame(resultats)
plancher

## Contrôles

In [ ]:
# 1) La constante est-elle stable d'un pli à l'autre ?
for s in ["mean", "median"]:
    cs = constantes[s]
    print(s, ": constante par pli ->", [round(c, 1) for c in cs], "| écart max :", round(max(cs) - min(cs), 1))

# 2) Comparaison avec la mesure « sur tout le train » de 1-10
mae_moy_tout = (y - y.mean()).abs().mean()
mae_med_tout = (y - y.median()).abs().mean()
print("MAE moyenne  : tout le train", round(mae_moy_tout, 2), "| validation croisée", plancher.loc[0, "mae_cv"])
print("MAE médiane  : tout le train", round(mae_med_tout, 2), "| validation croisée", plancher.loc[1, "mae_cv"])
print("énoncé : 1807,09 pour la moyenne")

# 3) Quel plancher retenir ?
meilleur = plancher.sort_values("mae_cv").iloc[0]
print("plancher retenu :", meilleur["constante"], "->", meilleur["mae_cv"])

## Rapport

In [ ]:
R = ["# 2.3 Plancher de contrôle",
     f"- découpage : {N_SPLITS} plis stratifiés sur les déciles de loss, graine {SEED}",
     f"- MAE moyenne (CV) : {plancher.loc[0, 'mae_cv']:,} (± {plancher.loc[0, 'ecart_type']:,})",
     f"- MAE médiane (CV) : {plancher.loc[1, 'mae_cv']:,} (± {plancher.loc[1, 'ecart_type']:,})",
     f"- plancher retenu : {meilleur['constante']} -> {meilleur['mae_cv']:,}"]
(OUT / "p3_2-3_plancher_controle.md").write_text("\n".join(R), encoding="utf-8")
print("\n".join(R))

## Conclusion 2.3 *(à rédiger avec tes mots)*

> Sans modèle, la valeur à annoncer est ... (moyenne ou médiane ?) car la métrique est la MAE. Le plancher en validation vaut ... (± ...). C'est le score que tout modèle doit battre. La constante est stable d'un pli à l'autre (écart max ...), donc la mesure est fiable.